# Étape 3 — Première LoRA

**Projet 5 · LoRA Zellige** — Google Colab, GPU T4.

Objectif volontairement modeste : obtenir **une** LoRA qui fonctionne, de
bout en bout. Les comparaisons entre rangs et nombres de pas sont l'affaire
de l'Étape 4.

Le notebook commence par un **test rapide de 20 pas** qui valide toute la
chaîne en deux minutes. S'il passe, l'entraînement complet passera aussi.

> **Exécution → Modifier le type d'exécution → GPU T4**
> puis **Exécution → Tout exécuter**


## 1. Environnement


In [ ]:
# torchao 0.10 est preinstalle sur Colab et peft exige >= 0.16 :
# sans cette desinstallation, le branchement de la LoRA echoue sur
# ImportError: Found an incompatible version of torchao.
# Le projet ne s'en sert pas.
!pip -q uninstall -y torchao 2>&1 | tail -1

!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors bitsandbytes lpips pyyaml 2>&1 | tail -2

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())


## 2. Dépôt et jeu de données


In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet 2>/dev/null || (cd /content/projet && git fetch -q && git reset -q --hard origin/main)
%cd /content/projet
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py


In [ ]:
# Controle : la suite n'a de sens que si le jeu est complet
from pathlib import Path
n = len(list(Path('data/processed/train').glob('*.jpg')))
assert n == 29, f'{n} images au lieu de 29 : relancez la cellule precedente'
assert Path('data/processed/train/metadata.jsonl').exists(), 'legendes manquantes'
print(f'OK : {n} images et leurs legendes')


## 3. Test rapide — 20 pas

Deux minutes pour vérifier que tout s'enchaîne : branchement de la LoRA,
circulation des gradients, sauvegarde des poids, et rechargement par le
pipeline de génération.

C'est la cellule qui doit échouer si quelque chose cloche — pas celle qui
dure vingt minutes.


In [ ]:
!python src/train_lora.py --config configs/lora_r8.yaml --pas 20 --sortie runs/test 2>&1 | tail -25


In [ ]:
from pathlib import Path
poids = list(Path('runs/test').glob('*.safetensors'))
assert poids, 'aucun fichier de poids produit : lisez l erreur ci-dessus'
print('poids ecrits :', poids[0], f'({poids[0].stat().st_size/1e6:.1f} Mo)')


In [ ]:
# La LoRA se recharge-t-elle vraiment dans le pipeline ?
!python src/generate.py --lora runs/test --intensite 1.0 \
    --sortie results/test_rapide --rapide 2>&1 | tail -8


In [ ]:
from PIL import Image
from pathlib import Path
img = sorted(Path('results/test_rapide').glob('*.png'))
assert img, 'aucune image generee'
print('chaine complete validee')
display(Image.open(img[0]).resize((320, 320)))


## 4. Entraînement complet

Ce qui se passe à chaque pas :

1. le **VAE** compresse l'image : 512×512×3 → 64×64×4, soit 48 fois moins
2. on y ajoute du bruit, d'une intensité tirée au hasard
3. le **U-Net** prédit le bruit ajouté, en lisant la légende
4. on corrige les matrices LoRA à partir de l'écart

Tout le modèle est **gelé** : environ **1,5 million de paramètres entraînés
sur 860 millions**, soit 0,2 %.

Rang 8, 1000 pas, lot effectif 4. Comptez 15 à 25 minutes. Un point de
contrôle est écrit tous les 250 pas — Colab coupe sans prévenir.


In [ ]:
!python src/train_lora.py --config configs/lora_r8.yaml


In [ ]:
from pathlib import Path
assert Path('runs/r8/entrainement.json').exists(), \
    'entrainement interrompu : lisez l erreur ci-dessus'
print('entrainement termine')


## 5. La courbe de perte

Elle doit **descendre puis se stabiliser**. Plate = taux d'apprentissage
trop faible ; très agitée = trop fort.

Attention : une perte qui baisse ne prouve pas que le style est appris.
Elle dit seulement que le modèle prédit mieux le bruit. Le vrai juge, ce
sont les mesures de la section 7.


In [ ]:
import json, matplotlib.pyplot as plt

rapport = json.load(open('runs/r8/entrainement.json'))
courbe = rapport['courbe']
plt.figure(figsize=(9, 3.5))
plt.plot([c['pas'] for c in courbe], [c['perte'] for c in courbe])
plt.xlabel('pas'); plt.ylabel('perte (MSE sur le bruit)')
plt.title(f"{rapport['nom']} - perte finale {rapport['perte_finale']}")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

for cle in ['parametres_entrainables', 'part_entrainable_pct', 'duree_s',
            'duree_par_pas_s', 'vram_pic_go', 'taille_lora_mo']:
    print(f'{cle:<26} {rapport[cle]}')


## 6. Générer avec la LoRA

Mêmes 10 prompts, mêmes 4 graines que la baseline. Rien d'autre ne change,
donc tout écart vient de la LoRA et d'elle seule.


In [ ]:
!python src/generate.py --lora runs/r8 --intensite 1.0 --sortie results/r8
!python src/evaluate.py --dossier results/r8 --nom lora_r8_1000pas


## 7. Avant / après

La figure la plus importante du rapport. Même prompt, même graine,
seule la LoRA diffère.

Les 40 images de la baseline ne sont pas dans Git — trop lourdes, et
le sujet l'interdit. Seules les **mesures** y sont. Une nouvelle
session Colab doit donc les régénérer.

Ce n'est pas un problème : la génération est déterministe (mêmes
prompts, mêmes graines, même ordonnanceur), donc on retrouve
exactement les mêmes images qu'à l'Étape 2. C'est précisément à quoi
sert d'avoir figé les graines.


In [ ]:
# Regenere la baseline seulement si elle manque (~3 min 30)
import subprocess
from pathlib import Path

if Path('results/baseline/D1_s1234.png').exists():
    print('images de la baseline deja presentes')
else:
    print('images de la baseline absentes, regeneration...', flush=True)
    code = subprocess.run('python src/generate.py --sortie results/baseline',
                          shell=True).returncode
    assert code == 0, 'la regeneration de la baseline a echoue'


In [ ]:
from PIL import Image
import yaml, matplotlib.pyplot as plt

prompts = yaml.safe_load(open('configs/eval_prompts.yaml'))['prompts']
graine = 1234
fig, axes = plt.subplots(2, len(prompts), figsize=(2.6 * len(prompts), 6.2))
for col, p in enumerate(prompts):
    for lig, dossier in enumerate(['results/baseline', 'results/r8']):
        ax = axes[lig, col]
        ax.imshow(Image.open(f"{dossier}/{p['id']}_s{graine}.png")); ax.axis('off')
        if lig == 0: ax.set_title(f"{p['id']}\n{p['categorie']}", fontsize=9)
fig.text(0.004, 0.72, 'AVANT', rotation=90, fontsize=12, weight='bold')
fig.text(0.004, 0.26, 'APRES', rotation=90, fontsize=12, weight='bold')
plt.suptitle('Baseline SD 1.5 vs LoRA rang 8 - graine 1234', fontsize=13)
plt.tight_layout(); plt.savefig('results/figures/avant_apres_r8.png', dpi=110)
plt.show()


## 8. Les chiffres


In [ ]:
import pandas as pd
df = pd.read_csv('results/results.csv').set_index('nom')
colonnes = ['style_dino_moy', 'style_clip_moy', 'prompt_domaine',
            'prompt_transfert', 'prompt_hors_domaine',
            'memorisation_max_dino', 'diversite_lpips']
t = df[colonnes].T
if 'lora_r8_1000pas' in t.columns:
    t['ecart'] = (t['lora_r8_1000pas'] - t['baseline']).round(4)
t


### Comment lire cet écart

| Ligne | Attendu | Pourquoi |
|---|---|---|
| `style_dino_moy` | **nette hausse** depuis 0,274 | preuve que le style est appris |
| `prompt_domaine` | baisse modérée tolérée | la LoRA pousse son style au détriment du texte |
| `prompt_transfert` | ne doit pas s'effondrer | sinon le modèle recopie au lieu d'avoir appris un style |
| `prompt_hors_domaine` | **stable** vers 33,1 | une chute = le modèle de base est abîmé |
| `memorisation_max_dino` | proche de 0,737 | au-delà, suspicion de recopie |
| `diversite_lpips` | ne doit pas s'effondrer | sinon les 4 graines donnent la même image |


## 9. Sauvegarder


In [ ]:
from google.colab import userdata
jeton = userdata.get('GITHUB_TOKEN')
url = f'https://{jeton}@github.com/HatimTj/lora-zellige-patrimoine.git'

!git config user.name 'HatimTj'
!git config user.email 'hatimtajimi1@gmail.com'
!git add results/results.csv results/r8/manifeste.json results/r8/evaluation.json \
         runs/r8/entrainement.json results/figures/avant_apres_r8.png
!git commit -m 'etape3: premiere LoRA rang 8, 1000 pas'

# pull --rebase d'abord : le depot a pu bouger depuis le clone de cette
# session, et un push sans cela est rejete (fetch first). Le rebase replace
# le commit au-dessus des autres, l'historique reste lineaire.
!git pull --rebase -q {url} main
!git push {url} main


> Après exécution : clic droit sur la cellule ci-dessus → **Effacer la
> sortie**. En cas d'échec, git affiche l'URL du dépôt, jeton compris.


## 10. Récupérer la LoRA

Gardez ce fichier : il sert à la démo de l'Étape 6, et le réentraîner coûte
20 minutes de GPU.


In [ ]:
from google.colab import files
files.download('runs/r8/pytorch_lora_weights.safetensors')
